# Mock Interview Practice — Day 27
**FAANG / Top Product Company Level — Week 4, Day 6** · Wed Nov 18, 2026

*Day 26 closed the synthesis stretch (Wilson bounds, Benford screens, scheduler internals, incident command). Day 27 pushes the remaining frontier: inequality statistics that rank customers fairly, ad-auction mechanics with budgets, classic algorithms interviewers still ask (Fenwick trees, Rabin-Karp), statistically sound distributed computing (bootstrap, PageRank), Catalyst plan forensics, timestamp correctness at ingestion, a Kappa replay harness, and the governance/monitoring trio (external egress, queueing theory, critical paths, tail SLOs, auto-triage).*

**18 questions · 8 categories · 11 [FAANG] / 7 [HARD]** — every solution cell is runnable top-to-bottom and asserts its own correctness.

| # | Category | Topic | Tag |
|---|----------|-------|-----|
| Q1 | SQL Advanced | Pareto + Gini via Lorenz curve | [FAANG] |
| Q2 | SQL Advanced | Second-price ad auction + budget pacing | [FAANG] |
| Q3 | Python | Fenwick tree + inversion counting | [FAANG] |
| Q4 | Python | Rabin-Karp rolling hash | [FAANG] |
| Q5 | PySpark | Distributed bootstrap confidence intervals | [FAANG] |
| Q6 | PySpark | Distributed PageRank with dangling nodes | [FAANG] |
| Q7 | Spark Deep | Catalyst logical optimizer forensics | [HARD] |
| Q8 | Spark Deep | Physical planning: join strategy + exchanges | [HARD] |
| Q9 | ETL | Timestamp correctness: INT96, timezones, ANSI parser | [HARD] |
| Q10 | ETL | Databricks Voyager cross-platform migration gates | [FAANG] |
| Q11 | ETL | Kappa architecture: replay + dual-run | [HARD] |
| Q12 | Data Modeling | Marketplace catalog + effective-dated price book | [FAANG] |
| Q13 | Data Modeling | Experimentation platform data model | [HARD] |
| Q14 | Snowflake | External network access + API integrations | [HARD] |
| Q15 | Snowflake | Warehouse concurrency: queueing theory | [FAANG] |
| Q16 | Monitoring | Orchestration: DAG critical path + slack + backfill | [FAANG] |
| Q17 | Monitoring | Tail-latency SLOs for data products | [FAANG] |
| Q18 | Monitoring | Auto-triage: freshness alert → root cause | [HARD] |

## Q1 — SQL Advanced: Pareto + Gini via the Lorenz Curve [FAANG]

**Interviewer (Meta, monetization analytics):** Leadership asks for "one number" for how unequal our customer spend is, plus the classic Pareto check. Raw top-N share hides the shape; the Gini coefficient captures it — but you must derive it in SQL, not hand-wave the formula.

**Requirements**

1. From `spend` rows (customer, spend), compute the **Lorenz curve**: customers sorted ascending by spend, cumulative spend share `L` vs cumulative population share.
2. Gini = `1 − 2A`, where `A` is the trapezoid area under the Lorenz curve. Implement the trapezoid sum with a `lag` window.
3. Compute the **Pareto share**: fraction of total spend held by the top 20% of customers.
4. Verify Gini independently with the closed form: `G = (2·Σ i·xᵢ − (n+1)·Σx) / (n·Σx)` over ascending `x` — the two must agree to 1e-9.

**Follow-ups to be ready for:** why trapezoid and not rectangles (rectangles over/under-shoot for discrete points), what Gini = 0 / Gini = 1 mean, and why a Gini on revenue differs from one on orders.

In [0]:
# Solution Q1 — Pareto + Gini via Lorenz trapezoid (validated vs closed form)
from pyspark.sql import functions as F, Window

spend_rows = [("c1", 5000.0), ("c2", 3200.0), ("c3", 1800.0), ("c4", 900.0),
               ("c5", 700.0), ("c6", 500.0), ("c7", 400.0), ("c8", 300.0),
               ("c9", 200.0), ("c10", 100.0)]
df = spark.createDataFrame(spend_rows, ["customer", "spend"]).cache()
total = sum(v for _, v in spend_rows)
n = len(spend_rows)

# Lorenz curve: ascending sort, cumulative share vs population share
w_asc = Window.orderBy(F.col("spend").asc())
lorenz = (df.withColumn("i", F.row_number().over(w_asc))
            .withColumn("cum", F.sum("spend").over(w_asc))
            .withColumn("L", F.col("cum") / F.lit(total))
            .withColumn("L_prev", F.lag("L", 1, 0.0).over(w_asc)))

# trapezoid area: A = (1/n) * Σ (L_prev + L)/2
A = lorenz.select(F.sum((F.col("L_prev") + F.col("L")) / 2).alias("s")).first()["s"] / n
gini = 1 - 2 * A

# independent check: closed form over ascending values
xs = sorted(v for _, v in spend_rows)
gini_ref = (2 * sum((i + 1) * x for i, x in enumerate(xs)) - (n + 1) * total) / (n * total)
assert abs(gini - gini_ref) < 1e-9, (gini, gini_ref)
assert 0 < gini < 1

# Pareto: top 20% of customers' spend share
w_desc = Window.orderBy(F.col("spend").desc())
pareto = (df.withColumn("rank", F.row_number().over(w_desc))
            .where(F.col("rank") <= n // 5)
            .agg(F.sum("spend").alias("s")).first()["s"]) / total
assert abs(float(pareto) - (5000 + 3200) / total) < 1e-9 and float(pareto) > 0.5

lorenz.show(5)
print(f"Q1 OK — Gini={gini:.4f} (closed form {gini_ref:.4f}) · top-20% spend share={float(pareto):.2%}")

# TALK TRACK
# 1) Window functions: row_number for population share, running sum for Lorenz point, lag for trapezoid.
# 2) lag(..., 1, 0.0) gives L_0 = 0 — the curve starts at the origin; missing that inflates A.
# 3) Gini is scale-invariant (doubling every spend changes nothing) but NOT rank-invariant.
# 4) At scale: replace the single global window with a two-stage sum (partition-local cumsum,
#    then merge) to avoid one-task skew.

## Q2 — SQL Advanced / Algorithmic: Second-Price Ad Auction with Budget Pacing [FAANG]

**Interviewer (Google, ads):** We run second-price auctions for a banner slot. Bidders have a private valuation-bid and a **remaining budget**. Each auction: the highest *eligible* bidder wins and pays the max of the reserve price and the second-highest eligible bid — but never more than their own remaining budget. A bidder with zero budget is ineligible. Model this precisely and prove the edge cases.

**Requirements**

1. Implement `run_auction(bidders, reserve)` returning `(winner, price_paid)`.
2. Handle: single eligible bidder (pays reserve), budget-capped payment, exhausted bidders excluded, no eligible bidder (no winner).
3. Debit the winner's budget in place; simulate a sequence of auctions and assert every intermediate state.

**Follow-ups:** why second-price is truthful (dominant strategy = bid your valuation), how budget pacing differs from bid shading, and what changes with a first-price auction (overbidding risk, bid shading becomes essential).

In [0]:
# Solution Q2 — second-price auction with budget pacing (all edge cases asserted)
class Bidder:
    def __init__(self, name, bid, budget):
        self.name, self.bid, self.budget = name, bid, budget
    def eligible(self):
        return self.budget > 0

def run_auction(bidders, reserve):
    elig = [b for b in bidders if b.eligible() and b.bid >= reserve]
    if not elig:
        return None, 0.0
    elig_sorted = sorted(elig, key=lambda b: -b.bid)
    winner = elig_sorted[0]
    price = reserve if len(elig_sorted) == 1 else max(reserve, elig_sorted[1].bid)
    price = min(price, winner.budget)      # never pay more than remaining budget
    winner.budget -= price
    return winner.name, price

bidders = [Bidder("alpha", 5.0, 40.0), Bidder("beta", 4.0, 60.0), Bidder("gamma", 3.0, 100.0)]

assert run_auction(bidders, 1.0) == ("alpha", 4.0)          # winner pays beta's bid, not his own
assert run_auction(bidders, 1.0) == ("alpha", 4.0)          # alpha budget 40 -> 32
assert abs(bidders[0].budget - 32.0) < 1e-9
bidders[0].budget = 1.5                                      # near exhaustion
assert run_auction(bidders, 1.0) == ("alpha", 1.5)          # capped at remaining budget
assert bidders[0].budget == 0
assert run_auction(bidders, 1.0) == ("beta", 3.0)           # alpha now ineligible; beta pays gamma's 3.0
assert run_auction(bidders, 9.0) == (None, 0.0)             # reserve above every bid
print("Q2 OK — second price + pacing: budget-capped payment, exhausted bidders excluded, no-winner case")

# TALK TRACK
# 1) Truthfulness: in a pure second-price auction your dominant strategy is bidding your true
#    valuation — your price is set by the NEXT bidder, so over/under-bidding only changes whether
#    you win unprofitably or lose profitably. Budget caps break the clean equilibrium: a capped
#    winner pays less than second price, which is why real systems blend in pacing (throttling
#    eligibility across the day) rather than hard caps alone.
# 2) Tie-breaking (equal bids) must be deterministic (e.g., bidder id) or the auction is not
#    replayable — a classic incident source.

## Q3 — Python: Fenwick Tree (BIT) and Inversion Counting [FAANG]

**Interviewer (Amazon):** A merchandising team wants to know how "out of order" a price feed is — the number of pairs `(i, j)` with `i < j` but `price[i] > price[j]`. The feed is a million rows; the naive O(n²) count dies. Build a **Fenwick tree** (binary indexed tree), use it to count inversions in O(n log n), and prove it matches brute force.

**Requirements**

1. Implement `bit_update` / `bit_query` with the `i & (-i)` lowbit trick (1-indexed internally).
2. `count_inversions(arr)`: coordinate-compress values, sweep right-to-left, count already-seen strictly smaller elements.
3. Prove the BIT's prefix-sum property on `[1,2,3,4,5]`, then verify inversions against O(n²) brute force on edge cases AND a random 2,000-element array.

**Follow-ups:** why sweep right-to-left (or left-to-right counting greater elements — both work, pick one and be consistent), duplicate handling (strictly smaller → equal values are NOT inversions), and how this generalizes to counting range-sum queries in O(log n).

In [0]:
# Solution Q3 — Fenwick tree + inversion counting (validated vs brute force incl. 2000-element case)
import numpy as np

def bit_update(bit, i, delta, n):
    i += 1                      # internal 1-indexing
    while i <= n:
        bit[i - 1] += delta
        i += i & (-i)           # climb to next responsible node

def bit_query(bit, i):          # sum of elements [0..i]
    s = 0
    i += 1
    while i > 0:
        s += bit[i - 1]
        i -= i & (-i)           # strip lowbit
    return s

def count_inversions(arr):
    n = len(arr)
    rank = {v: r for r, v in enumerate(sorted(set(arr)))}   # coordinate compression
    bit = [0] * n
    inv = 0
    for x in reversed(arr):     # right-to-left sweep
        r = rank[x]
        if r > 0:
            inv += bit_query(bit, r - 1)   # strictly smaller values already seen
        bit_update(bit, r, 1, n)
    return inv

# edge cases vs brute force
for t in [[3, 1, 2], [5, 4, 3, 2, 1], [1, 2, 3, 4], [2, 7, 1, 5, 3], [7, 7, 7], []]:
    brute = sum(1 for i in range(len(t)) for j in range(i + 1, len(t)) if t[i] > t[j])
    assert count_inversions(t) == brute, t

# random 2000-element stress test
rng = np.random.default_rng(0)
big = rng.integers(0, 1000, 2000).tolist()
assert count_inversions(big) == sum(1 for i in range(len(big))
                                    for j in range(i + 1, len(big)) if big[i] > big[j])

# prefix-sum property of the BIT itself
bit = [0] * 5
for i, v in enumerate([1, 2, 3, 4, 5]):
    bit_update(bit, i, v, 5)
assert [bit_query(bit, i) for i in range(5)] == [1, 3, 6, 10, 15]
print("Q3 OK — BIT prefix sums exact; inversions match brute force everywhere")

# TALK TRACK
# 1) O(n log n): n updates + n queries, each O(log n) after coordinate compression (raw values
#    could be huge; ranks are 0..n-1).
# 2) Reversed sweep: when we reach x, the BIT holds everything to its RIGHT; querying rank-1
#    counts right-side elements strictly smaller — exactly an inversion pair with x on the left.
# 3) Duplicates: strictly-smaller query means [7,7,7] has zero inversions — say this out loud,
#    interviewers probe it.
# 4) PySpark angle: per-partition BIT + merge gives approximate global ordering stats; exact
#    inversions need a sort-based approach (count via merge sort or a global BIT on a driver).

## Q4 — Python: Rabin-Karp Rolling Hash [FAANG]

**Interviewer (Microsoft):** Find every occurrence of a pattern in a large document. The API team needs substring search over millions of short patterns; `str.find` is fine for one pattern but explain the **rolling hash** approach — its O(n+m) expected time, its collision behavior, and why we still verify on hash match.

**Requirements**

1. Implement `rabin_karp(text, pattern)` with polynomial rolling hash: `hash = Σ ord(c)·base^(m-1-i) mod p`.
2. Roll the window in O(1) per shift: subtract the outgoing character's contribution, multiply, add the incoming one.
3. Verify the actual substring when hashes match (Las Vegas, not Monte Carlo).
4. Validate against Python's `str.find` loop on edge cases and a 4,000-character random scan with an embedded needle.

**Follow-ups:** expected vs worst-case complexity (adversarial collisions → O(n·m)), why the modulus should be a large prime, double hashing to slash collision probability, and where Rabin-Karp beats KMP (multiple patterns of equal length, e.g. plagiarism detection).

In [0]:
# Solution Q4 — Rabin-Karp with verified rolling hash (validated vs str.find)
import numpy as np

def rabin_karp(text, pattern, base=256, mod=10**9 + 7):
    n, m = len(text), len(pattern)
    if m == 0 or m > n:
        return []
    hp = ht = 0
    h = pow(base, m - 1, mod)                  # base^(m-1) mod p for the outgoing char
    for i in range(m):
        hp = (hp * base + ord(pattern[i])) % mod
        ht = (ht * base + ord(text[i])) % mod
    hits = []
    for s in range(n - m + 1):
        if ht == hp and text[s:s + m] == pattern:   # verify: guard against collisions
            hits.append(s)
        if s < n - m:                                 # roll the hash
            ht = ((ht - ord(text[s]) * h) * base + ord(text[s + m])) % mod
    return hits

for t, p in [("abracadabra", "abra"), ("aaaaaa", "aaa"), ("hello world", "xyz"),
             ("mississippi", "issi"), ("abc", ""), ("abc", "abcd")]:
    expect = []
    if p:
        s0 = t.find(p)
        while s0 != -1:
            expect.append(s0)
            s0 = t.find(p, s0 + 1)
    assert rabin_karp(t, p) == expect, (t, p)

rng = np.random.default_rng(1)
text = ("".join(rng.choice(list("ab"), size=3000).tolist()) + "needle"
        + "".join(rng.choice(list("ab"), size=1000).tolist()))
assert rabin_karp(text, "needle") == [3000]
print("Q4 OK — rolling-hash matches str.find on all cases + 4000-char scan")

# TALK TRACK
# 1) Rolling step: subtract ord(outgoing)*base^(m-1), THEN multiply by base and add incoming —
#    order matters; the modulo after every step keeps numbers small.
# 2) Verification makes it Las Vegas: a hash match triggers an m-char comparison, so false
#    positives cost time but never wrongness.
# 3) Collision math: P(false hit) ≈ 1/p per comparison — with p ≈ 1e9, negligible; adversaries
#    break that, hence double hashing in security contexts.
# 4) Multiple equal-length patterns: hash all patterns into a set, scan the text once —
#    Rabin-Karp's killer feature over KMP.

## Q5 — PySpark Coding: Distributed Bootstrap Confidence Intervals [FAANG]

**Interviewer (Netflix, experimentation platform):** A/B dashboards show point estimates only. Leadership wants 95% confidence intervals on the A/B revenue ratio — but the classic bootstrap is 2,000 resamples of the full data, and pandas on the driver won't scale. Distribute the bootstrap across executors **without collecting raw rows to the driver**.

**Requirements**

1. Resample-and-recompute must happen per-partition via `mapInPandas` (Spark 4: the UDF receives an **iterator of pandas DataFrames**).
2. Seeding: each rep gets a deterministic seed derived from its id, so replays are reproducible; the base data generation must be IDENTICAL on driver and executors (same RNG sequence) or the CI describes different data than the point estimate.
3. Aggregate the 2,000 bootstrap ratios with `percentile_approx`; verify (a) the point estimate lies inside the CI, (b) the CI width is within 40% of the delta-method (CLT) approximation, (c) the rep count is exact.

**Follow-ups:** why `mapInPandas` over a UDF (vectorized, Arrow batched), bias-corrected-and-accelerated (BCa) vs percentile intervals, and why bootstrap beats delta-method for skewed metrics like ratio-of-means.

In [0]:
# Solution Q5 — distributed bootstrap CIs via mapInPandas (iterator-of-DataFrames signature)
import numpy as np
import pandas as pd
from pyspark.sql import functions as F, types as T

def gen_data():
    # EXACT same sequence on driver and inside the UDF: one RNG stream, A then B.
    r = np.random.default_rng(42)
    va = np.maximum(r.gamma(4.0, 2.5, 150), 1.0).round(2)
    vb = np.maximum(r.gamma(4.0, 2.2, 150), 1.0).round(2)
    return va, vb

vals_A, vals_B = gen_data()
nA = nB = 150
point_ratio = float(vals_A.mean() / vals_B.mean())

# driver-side point estimate must agree with the distributed data
users = pd.DataFrame({"user": [f"a{i}" for i in range(nA)] + [f"b{i}" for i in range(nB)],
                      "group": ["A"] * nA + ["B"] * nB,
                      "value": np.concatenate([vals_A, vals_B])})
users_sdf = spark.createDataFrame(users)
spark_point = (users_sdf.groupBy("group").agg(F.expr("avg(value)").alias("m"))
               .groupBy().agg(F.expr("max(case when group='A' then m end)"
                                     " / max(case when group='B' then m end)").alias("ratio"))
               .first()["ratio"])
assert abs(float(spark_point) - point_ratio) < 1e-9

N_REPS = 2000

def bootstrap_batch(batches):
    # Spark 4: mapInPandas UDFs take an ITERATOR of pandas DataFrames and yield DataFrames.
    va, vb = gen_data()                      # identical data-generation sequence as driver
    for pdf in batches:
        out = []
        for rep in pdf["rep"]:
            rs = np.random.default_rng(int(rep) * 7919)   # deterministic per-rep seed
            sa = rs.choice(va, size=len(va), replace=True)
            sb = rs.choice(vb, size=len(vb), replace=True)
            out.append({"rep": int(rep), "ratio": float(sa.mean() / sb.mean())})
        yield pd.DataFrame(out)

schema = T.StructType([T.StructField("rep", T.LongType()), T.StructField("ratio", T.DoubleType())])
reps = spark.range(N_REPS).withColumnRenamed("id", "rep").repartition(16)
boot = reps.mapInPandas(bootstrap_batch, schema)
n_boot = boot.count()
stats = boot.agg(F.percentile_approx("ratio", 0.025).alias("lo"),
                 F.percentile_approx("ratio", 0.975).alias("hi")).first()
lo, hi = float(stats["lo"]), float(stats["hi"])

assert n_boot == N_REPS
assert lo < point_ratio < hi, (lo, point_ratio, hi)
mA, mB = vals_A.mean(), vals_B.mean()
clt_se = np.sqrt(vals_A.var(ddof=1) / (nA * mA**2) + vals_B.var(ddof=1) / (nB * mB**2))
clt_half = 1.96 * clt_se                      # delta-method half-width for the ratio
assert abs((hi - lo) / 2 - clt_half) / clt_half < 0.40
print(f"Q5 OK — point={point_ratio:.4f} · boot CI=({lo:.4f},{hi:.4f}) · delta-method half-width={clt_half:.4f}")

# TALK TRACK
# 1) The trap that cost this cell two debugging rounds: the UDF originally regenerated data with
#    a DIFFERENT seed than the driver — the point estimate sat outside its own CI. The base data
#    must be bit-identical on both sides; embed literals or replicate the exact RNG call sequence.
# 2) Spark 4 API: mapInPandas UDFs accept an iterator of DataFrames and yield DataFrames — a
#    plain function(batch)->df raises a confusing generator error.
# 3) numpy .var()'s SECOND POSITIONAL arg is axis, not ddof — use var(ddof=1) or CI math drifts.
# 4) percentile_approx is a distributed percentile; for exact order statistics use
#    percentile(...) (needs sort) or accept the approximation trade-off.

## Q6 — PySpark Coding: Distributed PageRank with Dangling Nodes [FAANG]

**Interviewer (Google, search infra — the classic):** Implement PageRank as a distributed DataFrame join-and-aggregate loop: rank diffusion over edges, **dangling-node mass** (nodes with no out-links leak probability and must be redistributed), and L1-convergence detection. No GraphFrames library — plain PySpark only.

**Requirements**

1. Each iteration: contribute `rank/deg` along out-edges, sum per destination, add redistributed dangling mass, apply damping `d = 0.85`: `rank' = (1-d)/N + d·(Σcontribs + danglingMass/N)`.
2. Iterate until L1 change < 1e-6 (cap 40 iterations — every iteration is Spark jobs; tight tolerances cost minutes on Spark Connect).
3. Validate against an independent pure-Python reference to 1e-4 and check ranks sum to 1.

**Follow-ups:** why dangling redistribution matters (without it total probability leaks away — ranks shrink each iteration), convergence rate vs damping factor, and how the join pattern maps to billion-edge graphs (cost per iteration, partitioning by node id, precomputed out-degrees).

In [0]:
# Solution Q6 — PageRank on DataFrames with dangling redistribution (validated vs reference)
from pyspark.sql import functions as F

edges_list = [("n1","n2"),("n1","n3"),("n2","n3"),("n2","n4"),("n3","n5"),
              ("n4","n5"),("n4","n6"),("n5","n6"),("n5","n1"),("n7","n3"),
              ("n3","n7")]
nodes = sorted({x for e in edges_list for x in e})
N = len(nodes)

def pagerank_spark(edges_list, d=0.85, tol=1e-6, max_iter=40):
    edges = spark.createDataFrame(edges_list, ["src", "dst"])
    out_deg = edges.groupBy("src").agg(F.count("dst").alias("deg"))
    edges = edges.join(out_deg, "src")
    nodes_df = spark.createDataFrame([(n,) for n in nodes], ["node"])
    ranks = nodes_df.withColumn("rank", F.lit(1.0 / N))
    for it in range(max_iter):
        contribs = (ranks.join(edges, ranks["node"] == edges["src"])
                    .withColumn("c", F.col("rank") / F.col("deg"))
                    .groupBy("dst").agg(F.sum("c").alias("s")))
        # dangling mass: rank of nodes with no out-links, redistributed uniformly
        danglers = (ranks.join(out_deg, ranks["node"] == out_deg["src"], "left_anti")
                   .agg(F.sum("rank").alias("dr")).first()["dr"]) or 0.0
        new_ranks = (nodes_df.join(contribs, nodes_df["node"] == contribs["dst"], "left")
                     .withColumn("s", F.coalesce(F.col("s"), F.lit(0.0)))
                     .withColumn("rank", F.lit((1 - d) / N)
                                 + F.lit(d) * (F.col("s") + F.lit(danglers / N))))
        joined = ranks.join(new_ranks.withColumnRenamed("rank", "new_rank"), "node")
        l1 = joined.select(F.sum(F.abs(F.col("rank") - F.col("new_rank"))).alias("l1")).first()["l1"] or 0.0
        ranks = new_ranks.select("node", "rank")
        if l1 < tol:
            return ranks, it + 1
    return ranks, max_iter

pr_sdf, iters = pagerank_spark(edges_list)
pr_spark = {r["node"]: float(r["rank"]) for r in pr_sdf.collect()}

# independent pure-Python reference
M = {n: [] for n in nodes}
for s, t_ in edges_list:
    M[s].append(t_)
r = {n: 1.0 / N for n in nodes}
for _ in range(200):
    dangling = sum(r[n] for n in nodes if not M[n])
    nr = {n: (1 - 0.85) / N + 0.85 * (dangling / N
          + sum(r[s] / len(M[s]) for s in nodes if n in M[s])) for n in nodes}
    if sum(abs(nr[n] - r[n]) for n in nodes) < 1e-12:
        r = nr
        break
    r = nr

for n in nodes:
    assert abs(pr_spark[n] - r[n]) < 1e-4, (n, pr_spark[n], r[n])
assert abs(sum(pr_spark.values()) - 1.0) < 1e-4
best = max(r, key=r.get)
print(f"Q6 OK — {iters} iterations · top node {best} rank={pr_spark[best]:.4f} · Σrank={sum(pr_spark.values()):.6f}")

# TALK TRACK
# 1) left_anti join isolates dangling nodes (in ranks but not in out_deg) — the elegant one-liner
#    interviewers look for.
# 2) Convergence: L1 over ranks; tol 1e-6 converges in ~30 iterations at d=0.85. A 1e-9 tolerance
#    ran the FULL 40-iteration cap and timed out the cell — every iteration is a chain of Spark
#    jobs on Spark Connect; tune tolerance to what you actually need.
# 3) At scale: pre-partition edges by src hash, persist ranks each iteration, and swap the
#    convergence check to a sampled L1 (exact L1 costs an extra job per iteration).

## Q7 — Spark Optimization & Deep Spark: Catalyst Logical Optimizer Forensics [HARD]

**Interviewer (Databricks):** A query is slow. Before touching configs, read the plan: what did Catalyst's **logical optimizer** already do for you, and how do you prove it? Demonstrate predicate pushdown, projection pruning, and filter-into-join pushdown — then explain the one case where the plan text lies.

**Requirements**

1. Build `orders(order_id, region_id, amount, order_date)`; query `SELECT order_id FROM orders WHERE amount > 30000`. Prove in the physical plan that (a) a Filter node exists and (b) `order_date` never appears — projection pruning dropped the unused column.
2. Join orders to a regions dimension filtered by `zone = 'EMEA'`. Verify the result **semantically** (row count vs an independent recount) — and explain why the initial (non-final) AQE/Photon plan can display constraint filters that look wrong.
3. Use a plan-capture helper that works on Spark Connect (`_explain_string()` with a stdout-redirect fallback).

**Follow-ups:** which rules fire (PushDownPredicates, ColumnPruning, InferFiltersFromConstraints), why `isFinalPlan=false` plans are pre-execution sketches, and when pushdown stops (non-deterministic functions, UDAFs, certain outer joins).

In [0]:
# Solution Q7 — plan forensics: pushdown + pruning proven, semantics checked by recount
import io, contextlib, re
from pyspark.sql import functions as F

def get_plan(q):
    try:
        return q._explain_string()          # Spark Connect native
    except Exception:
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            q.explain()
        return buf.getvalue()

orders = spark.range(3000).selectExpr(
    "id as order_id", "cast(id % 50 as int) as region_id",
    "id * 12.5 as amount", "date_add(date'2026-01-01', cast(id % 90 as int)) as order_date")
orders.createOrReplaceTempView("d27_q7_orders")

# (1) filter + projection pruning
q = spark.sql("SELECT order_id FROM d27_q7_orders WHERE amount > 30000")
plan_txt = get_plan(q)
assert re.search(r"(Photon)?Filter", plan_txt), plan_txt      # filter survived/pushed
phys = plan_txt.split("== Physical Plan ==")[-1] if "== Physical Plan ==" in plan_txt else plan_txt
assert "order_date" not in phys, "projection pruning failed — unused column still scanned"
print("Q7a OK — Filter present; order_date pruned from the physical plan")

# (2) filter into a join — verified SEMANTICALLY, not by plan text
regions = spark.createDataFrame(
    [(i, f"region_{i}", "EMEA" if i % 2 == 0 else "APAC") for i in range(50)],
    ["region_id", "region_name", "zone"])
regions.createOrReplaceTempView("d27_q7_regions")
q2 = spark.sql("""
  SELECT o.order_id, r.region_name
  FROM d27_q7_orders o JOIN d27_q7_regions r ON o.region_id = r.region_id
  WHERE r.zone = 'EMEA'
""")
rows = q2.count()
exp_rows = orders.join(regions, "region_id").where("zone = 'EMEA'").count()   # independent recount
assert rows == exp_rows == 1500, (rows, exp_rows)
plan2 = get_plan(q2)
assert "BroadcastHashJoin" in plan2 or "HashJoin" in plan2
print(f"Q7b OK — join row count {rows} == independent recount {exp_rows}")
print("\n--- observed plan (initial, isFinalPlan=false) ---")
print(plan2[:800])

# TALK TRACK
# 1) The initial Photon/AQE plan displayed a stream-side constraint like
#    (region_id >= 0) AND (region_id = 0) AND (region_id <= 49) — which, taken literally, would
#    return 60 rows, not 1500. The plan was a PRE-EXECUTION sketch (isFinalPlan=false) whose
#    constraint display was misleading; the executed result was correct. LESSON: plan text is a
#    guide, row counts are proof. Always verify pushdown claims with an independent recount.
# 2) zone='EMEA' vanished from the build side because the LocalTableScan was already filtered
#    and zone projected away — for in-memory relations Catalyst folds filters into the local
#    table, so "no Filter node" can MEAN "pushdown worked". Read what is absent, not just what
#    is present.
# 3) Rule names to cite: PushDownPredicates, ColumnPruning, InferFiltersFromConstraints,
#    PushDownLeftSemiAntiJoin (semi/anti joins), EliminateSubqueryAliases.

## Q8 — Spark Optimization & Deep Spark: Physical Planning — Join Strategy + Exchange Requirements [HARD]

**Interviewer (Uber):** Same join, three plans: a 200k×150k self-ish join, the same join with a `MERGE` hint, and a fact-to-small-dimension join. Predict the **join strategy** each one gets, what forces a shuffle exchange, and which side becomes the broadcast build side — then verify against the actual plans and row counts.

**Requirements**

1. Show that join strategy is a **size** decision, not a row-count decision — and be ready for the surprise (150k tiny rows can still broadcast).
2. `MERGE` hint run must return exactly the same row count as the un-hinted run.
3. In the broadcast plan, identify `BuildRight` and explain why the small dimension is the build side.

**Follow-ups:** sort-merge vs shuffled-hash (memory vs sort spill), when AQE converts SMJ→BHJ at runtime (`spark.sql.adaptive.autoBroadcastJoinThreshold`), and why co-partitioning join keys across a pipeline kills the exchange entirely (bucketing — covered in day 10, name-drop it).

In [0]:
# Solution Q8 — join strategy + exchange forensics (counts verified three ways)
import io, contextlib, re
from pyspark.sql import functions as F

def get_plan(q):
    try:
        return q._explain_string()
    except Exception:
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            q.explain()
        return buf.getvalue()

big1 = spark.range(200_000).selectExpr("id as k1", "id as a1")
big2 = spark.range(150_000).selectExpr("id as k2", "id as b1")

# (1) big-big: expect a shuffle exchange... or a size-based surprise
joined = big1.join(big2, big1["k1"] == big2["k2"])
plan_large = get_plan(joined)
assert "Shuffle" in plan_large or "Exchange" in plan_large, plan_large
large_rows = joined.count()

# (2) same join, explicit MERGE hint — semantics must not change
joined_smj = big1.join(big2, big1["k1"] == big2["k2"]).hint("MERGE")
plan_smj = get_plan(joined_smj)
smj_rows = joined_smj.count()
assert smj_rows == large_rows == 150_000, (smj_rows, large_rows)

# (3) small dimension: broadcast hash join, build side on the right
dim = spark.createDataFrame([(i, f"dim_{i}") for i in range(50)], ["region_id", "region_name"])
facts = spark.range(100_000).selectExpr("id", "cast(id % 50 as int) as region_id")
fj = facts.join(dim, "region_id")
plan_bj = get_plan(fj)
assert "BroadcastHashJoin" in plan_bj, plan_bj
assert re.search(r"BuildRight|BuildLeft", plan_bj)
bj_rows = fj.count()
assert bj_rows == 100_000

print(f"Q8 OK — big-join rows={large_rows} · MERGE-hint rows={smj_rows} · broadcast rows={bj_rows}")
print("  strategies seen:")
print("   big-big :", sorted(set(re.findall(r"Photon?\w*?(?:BroadcastHashJoin|ShuffledHashJoin|SortMergeJoin)", plan_large))))
print("   MERGE   :", sorted(set(re.findall(r"Photon?\w*Join", plan_smj))))
print("   dim     :", re.findall(r"Photon\w*HashJoin[^\n]*", plan_bj))

# TALK TRACK
# 1) SURPRISE we observed: the 200k x 150k join got PhotonBroadcastHashJoin — because 150k
#    (long, long) rows is only ~2-3 MB, under the broadcast threshold. Broadcast is decided by
#    ESTIMATED SIZE, not row count. To force the shuffle-hash/sort-merge story, widen the rows
#    (string columns) or disable broadcast for that join — mentioning this shows you read plans
#    instead of reciting them.
# 2) The MERGE hint requested sort-merge; Photon may still override to a shuffled hash join when
#    its cost model prefers it — but the ROW COUNT is invariant. Strategy is an optimization;
#    correctness is a constraint. Never claim a hint guarantees a physical operator — verify.
# 3) BuildRight: the build side must fit in memory per task; Catalyst builds the smaller side.
#    In `facts.join(dim)` the dim (right operand) is built and broadcast — swapping the operands
#    flips BuildRight/BuildLeft.
# 4) Exchange appears because join keys must be co-partitioned: hash-partition both sides on the
#    key. Pre-partitioned/bucketed sources skip that shuffle — day 10's bucketing deep-dive.

## Q9 — ETL: Timestamp Correctness at Ingestion — INT96, Session Timezones, and the ANSI Parser [HARD]

**Interviewer (Airbnb):** A Parquet drop from a legacy Hive/Impala pipeline shows check-in times shifted by hours depending on which region reads it. Explain **INT96 vs INT64** timestamp encodings, prove what a "wall-clock string" means at parse time, and handle the strict parser on dirty data.

**Requirements**

1. Show `to_timestamp('2026-03-01T12:00:00Z')` == `to_timestamp('2026-03-01T12:00:00+00:00')` — offsets encode the instant — while an offset-less string is interpreted in the **session timezone**, rendering the same instant differently after `spark.sql.session.timeZone` changes.
2. Show that under ANSI mode `to_timestamp('not a date at all')` RAISES `CAST_INVALID_INPUT`, while `try_to_timestamp` returns NULL — and state the trap for out-of-range digit strings (they also raise, this environment: ANSI on).
3. Round-trip a timestamp through Parquet and assert the instant survives (Spark 3+ writes INT64 micros; talk INT96).

**Follow-ups:** INT96 stores local wall-clock with no offset (legacy readers reinterpret it in THEIR zone — the 8-hour-shift class of bug), `spark.sql.parquet.int96TimestampConversion`, and storing UTC + formatting at presentation as the invariant.

In [0]:
# Solution Q9 — timestamp semantics: offsets vs wall-clock, ANSI parser, Parquet round-trip
from pyspark.sql import functions as F

tz_conf = "spark.sql.session.timeZone"
orig_tz = spark.conf.get(tz_conf)

# (1) offset strings encode the SAME instant; offset-less strings depend on session tz
df = spark.sql("""SELECT
  to_timestamp('2026-03-01T12:00:00Z')  AS ts_utc,
  to_timestamp('2026-03-01T12:00:00+00:00') AS ts_off,
  to_timestamp('2026-03-01 12:00:00')   AS ts_wall
""")
row = df.first()
assert row["ts_utc"] == row["ts_off"], (row["ts_utc"], row["ts_off"])
print("offset forms equal:", row["ts_utc"], "| wall-clock (no offset):", row["ts_wall"])

try:
    spark.conf.set(tz_conf, "Asia/Kolkata")
    fmt_k = spark.sql("SELECT date_format(to_timestamp('2026-03-01T12:00:00Z'),"
                      " 'yyyy-MM-dd HH:mm:ss VV') AS s").first()["s"]
    spark.conf.set(tz_conf, orig_tz)
    fmt_0 = spark.sql("SELECT date_format(to_timestamp('2026-03-01T12:00:00Z'),"
                      " 'yyyy-MM-dd HH:mm:ss VV') AS s").first()["s"]
    assert fmt_k != fmt_0
    print(f"same instant, two zones: '{fmt_0}' -> '{fmt_k}' (session tz is settable)")
except Exception as e:
    print("tz conf not settable on this compute:", type(e).__name__)

# (2) ANSI strict parser: malformed input RAISES; try_to_timestamp tolerates
raised = False
try:
    spark.sql("SELECT to_timestamp('not a date at all') AS t").first()
except Exception:
    raised = True
safe = spark.sql("SELECT try_to_timestamp('not a date at all') AS t").first()["t"]
assert raised and safe is None, (raised, safe)
print("Q9a OK — malformed raises CAST_INVALID_INPUT under ANSI; try_to_timestamp -> NULL")

# (3) Parquet round-trip preserves the instant (INT64 microseconds in Spark 3+)
parq = "/Volumes/workspace/default/etl_test_volume/d27_q9_ts.parquet"
ts_df = spark.createDataFrame([("2026-03-01T12:00:00Z",)], ["s"]).withColumn("ts", F.expr("to_timestamp(s)"))
ts_df.write.mode("overwrite").parquet(parq)
back = spark.read.parquet(parq).first()["ts"]
orig = ts_df.first()["ts"]
assert back == orig, (back, orig)
print(f"Q9b OK — Parquet round-trip preserved instant: {back}")

# TALK TRACK
# 1) INT96 vs INT64 (talk track — Spark 3+ writes INT64 micros; you cannot produce INT96 here):
#    INT96 = 12 bytes = nanoseconds-of-day + julian day, written from LOCAL wall-clock with NO
#    offset. Legacy Hive/Impala readers then interpret those bytes in THEIR server timezone —
#    the "every region sees a different check-in time" bug. Spark reads INT96 either as UTC
#    (spark.sql.parquet.int96TimestampConversion=false) or converts assuming writer-local time
#    (true, the Hive-compat default). Migration playbook: read with BOTH settings, diff, then
#    normalize to UTC instants once at ingestion.
# 2) The invariant to state: store TIMESTAMPTZ instants (UTC), format to local wall-clock only at
#    presentation. The session timezone then changes RENDERING, never the data.
# 3) Parser policy: to_timestamp raises under ANSI (this workspace: ANSI on — both garbage AND
#    out-of-range digits raise); wrap dirty sources in try_to_timestamp + a quarantine flag, or
#    set an explicit format so '13:99' fails loudly in validation instead of silently nulling.
# 4) trap we hit live: try_to_timestamp existed alongside to_timestamp in the SAME query plan —
#    the safe path is one function swap, not a schema change.

## Q10 — ETL: Cross-Platform Migration with Databricks Voyager — Assessment, Wave Planning, and Cutover Gates [FAANG]

**Interviewer (Adobe, post-acquisition platform consolidation):** We're moving an acquired company's estate — 8 objects across Snowflake, Teradata, and BigQuery — onto the Lakehouse. Databricks Voyager automates the code/object conversion; your job is the **engineering judgment** around it: what to migrate first, how to score complexity, and what gates block cutover.

**Requirements**

1. Score each object: base weight by type (TABLE 1, VIEW 2, UDF 4 — UDFs carry logic risk) × size multiplier (>1e10 rows: 3, >1e8: 2, else 1).
2. Implement a **parity gate**: row count + a per-table `xxhash64` checksum over all columns must BOTH match between source and target — show it PASS on a faithful copy and BLOCK on value drift and on row-count mismatch.
3. Explain the wave plan: what order the scored inventory implies and why.

**Follow-ups:** why Voyager is best-effort on dialect-specific SQL (review flagged UDFs manually), what Voyager does NOT translate (stored procedures, scheduler semantics, RBAC), and dual-write vs backfill strategies for cutover.

In [0]:
# Solution Q10 — migration inventory scoring + checksum parity gates (all three gate paths verified)
from pyspark.sql import functions as F

inventory = [
    ("FINANCE", "GL_ENTRIES", "TABLE", 4.2e9, "TERADATA"),
    ("FINANCE", "GL_PERIOD_SUM", "VIEW", None, "TERADATA"),
    ("SALES", "ORDERS", "TABLE", 9.7e9, "SNOWFLAKE"),
    ("SALES", "ORDER_LINE", "TABLE", 2.4e10, "SNOWFLAKE"),
    ("SALES", "PRICE_UDF", "UDF", None, "SNOWFLAKE"),
    ("HR", "EMP_DIM", "TABLE", 1.1e6, "SNOWFLAKE"),
    ("HR", "EMP_HIST", "TABLE", 4.0e6, "SNOWFLAKE"),
    ("MARKETING", "CAMP_EVENTS", "TABLE", 6.5e10, "GOOGLE_BIGQUERY"),
]

def complexity_score(t, rows):
    base = {"TABLE": 1, "VIEW": 2, "UDF": 4}[t]
    size_mult = 3 if (rows or 0) > 1e10 else (2 if (rows or 0) > 1e8 else 1)
    return base * size_mult

# dtype trap hit live: mixing float row-counts and integer 0 in one column -> CANNOT_MERGE_TYPE
rows_pd = [(sch, obj, t, float(rows or 0), src, int(complexity_score(t, rows)))
           for sch, obj, t, rows, src in inventory]
scored = spark.createDataFrame(rows_pd, ["schema", "obj", "obj_type", "row_count", "source", "score"])

tables_n = scored.where("obj_type = 'TABLE'").count()
total_score = scored.agg({"score": "sum"}).first()["sum(score)"]
exp_total = sum(complexity_score(t, r) for _, _, t, r, _ in inventory)
assert tables_n == 6 and total_score == exp_total, (tables_n, total_score, exp_total)

scored.orderBy(F.col("score").desc(), "obj").show(truncate=False)

# parity gate: row count + checksum over ALL columns must both match
def checksum(df):
    return df.agg(F.count("*").alias("n"),
                  F.sum(F.xxhash64(*[c for c in df.columns])).alias("h")).first()

def gate_pass(src, tgt, name):
    s, t = checksum(src), checksum(tgt)
    ok = (s["n"] == t["n"]) and (int(s["h"]) == int(t["h"]))
    print(f"  gate {name}: rows {s['n']} vs {t['n']}, checksum {int(s['h'])} vs {int(t['h'])}"
          f" -> {'PASS' if ok else 'BLOCK'}")
    return ok

src = spark.createDataFrame([(1, "a", 10), (2, "b", 20), (3, "c", 30)], ["id", "k", "v"])
tgt_bad = src.withColumn("v", F.col("v") + 1)          # same rows, drifted values
assert gate_pass(src, src, "clean_target")
assert not gate_pass(src, tgt_bad, "value_drift")
assert not gate_pass(src, src.limit(2), "row_count_mismatch")
print("Q10 OK — scoring totals match derivation; all three gate paths behave")

# TALK TRACK
# 1) Wave plan implied by the scores: migrate the score-1 tail first (EMP_DIM, EMP_HIST) to
#    de-risk the pipeline pattern; the score-3 monsters (CAMP_EVENTS, ORDER_LINE) last, after
#    the toolchain is proven; PRICE_UDF (score 4 despite zero rows) early-ish because logic
#    conversion is the riskiest artifact and blocks the tables that depend on it.
# 2) Checksum gate: xxhash64 over ALL columns is order-independent for equality of MULTISETS —
#    but note it does not catch duplicate-row collapse on its own; pair it with the row count
#    (that is why BOTH must match). Sort-order and column-order differences need a stable
#    canonical projection before hashing.
# 3) Voyager reality: best-effort SQL conversion with flagged items; UDFs/stored procs and
#    scheduler/RBAC semantics stay manual. The gate model above is exactly what you wrap around
#    its output — Voyager converts, the checksum decides.
# 4) Cutover: dual-write window with parity gates nightly, backfill the delta, flip readers,
#    keep the source frozen-but-readable for rollback.

## Q11 — ETL: Kappa Architecture — Single Log, Versioned Logic, Dual-Run, and Replay [HARD]

**Interviewer (Stripe, payments data):** Your revenue rollup has a logic bug — REFUNDED amounts counted as revenue, and EUR never converted. Under Lambda you'd re-run a batch layer; under **Kappa** there is one immutable event log and every derived table is a pure function of `(log, logic_version)`. Demonstrate the full machinery: replay determinism, dual-run comparison, and late-arrival re-derivation.

**Requirements**

1. Build a small event log as a Delta table on a volume path (events: account, currency, amount, status).
2. `replay(version)`: v1 = the buggy logic (all amounts, no FX); v2 = the fix (COMPLETED only, EUR×1.1). Both are pure reads of the log.
3. Assert: v1/v2 outputs match independently-derived expectations; replaying v2 twice is byte-identical (determinism); appending one late event and re-deriving picks it up — with no separate batch layer.

**Follow-ups:** when Kappa breaks (non-additive logic like `count(distinct)` over all-time windows, enrichment against current dimension state), log compaction, and the dual-run gate (diff v1 vs v2, ship when the diff is fully explained).

In [0]:
# Solution Q11 — Kappa: one Delta log, two logic versions, deterministic replay + late append
from collections import defaultdict
from pyspark.sql import functions as F

EVENTS = [("e1", 100, "USD", 10.0, "COMPLETED"),
          ("e2", 100, "USD", 25.0, "COMPLETED"),
          ("e3", 100, "EUR", 40.0, "COMPLETED"),
          ("e4", 100, "USD", 99.0, "REFUNDED"),
          ("e5", 200, "USD", 7.5,  "COMPLETED")]
LOG = "/Volumes/workspace/default/etl_test_volume/d27_q11_log"
(spark.createDataFrame(EVENTS, ["event_id", "account", "currency", "amount", "status"])
 .write.format("delta").mode("overwrite").save(LOG))

def replay(version):
    log = spark.read.format("delta").load(LOG)          # pure read of the immutable log
    if version == 1:                                    # v1 — the bug: refunds counted, no FX
        return log.groupBy("account").agg(F.sum("amount").alias("revenue"))
    fx = F.when(F.col("currency") == "EUR", F.round(F.col("amount") * 1.1, 2)).otherwise(F.col("amount"))
    return (log.where("status = 'COMPLETED'").withColumn("usd", fx)
              .groupBy("account").agg(F.sum("usd").alias("revenue")))   # v2 — the fix

def as_dict(dfr):
    return {r["account"]: float(r["revenue"]) for r in dfr.collect()}

exp1 = defaultdict(float); exp2 = defaultdict(float)
for _, acct, cur, amt, st in EVENTS:
    exp1[acct] += amt
    if st == "COMPLETED":
        exp2[acct] += round(amt * 1.1 if cur == "EUR" else amt, 2)
v1, v2 = as_dict(replay(1)), as_dict(replay(2))
assert v1 == {a: float(e) for a, e in exp1.items()} and v2 == {a: float(e) for a, e in exp2.items()}, (v1, v2)
assert as_dict(replay(2)) == v2                           # determinism: same log + logic -> same output
print(f"Q11 OK — v1={dict(v1)} (bug: 174.0 includes 99 refund, EUR raw)  v2={dict(v2)} (fixed)")

# late arrival: append ONE event, re-derive — no batch layer, no backfill job
(spark.createDataFrame([("e6", 100, "USD", 5.0, "COMPLETED")],
                       ["event_id", "account", "currency", "amount", "status"])
 .write.format("delta").mode("append").save(LOG))
v3 = as_dict(replay(2))
exp3 = dict(exp2); exp3[100] = round(exp3[100] + 5.0, 2)
assert v3 == exp3, (v3, exp3)
print(f"Q11b OK — late append re-derived from the single log: {v3}")

# TALK TRACK
# 1) Dual-run gate (the interview core): run v1 and v2 side by side, diff, and REQUIRE every
#    diff to be explainable — here account 100 drops 174.0 -> 79.0 (refund out, EUR 44.0 in).
#    An unexplained diff row blocks the ship.
# 2) The derived table is a pure function of (log, version): determinism is what makes
#    "recompute everything" a valid recovery strategy — assert it (we did) rather than assume.
# 3) Where Kappa strains: non-additive logic (distinct counts) can't be fixed by naive replay
#    — needs windowed state or hybrid batch recompute; enrichment against CURRENT dimension
#    state breaks replay unless the enrichment is event-sourced too (as-of joins into the log).
# 4) Log compaction: keep raw events forever in cheap storage, compact aggregates upstream —
#    replay cost is bounded by log size, so compact aggressively.

## Q12 — Data Modeling: Multi-Seller Marketplace Catalog + Effective-Dated Price Book [FAANG]

**Interviewer (Flipkart):** Model a marketplace where the same physical product is sold by many sellers at different prices in different regional price books that change over time. Then answer the two questions PMs ask daily: "what is this listing's price today in the EU book?" and "how many sellers actively offer this variant?"

**Requirements**

1. Design the grain chain: `product` (brand/category) → `variant` (finish/voltage — sparse attributes) → `listing` (variant × seller × status) → `price_book_entry` (book, variant, seller, currency, price, effective_from/to).
2. Implement the **as-of join**: price where `effective_from <= as_of` AND (`effective_to IS NULL OR effective_to >= as_of`) — show a price that EXPIRED (old sellerA price) vs its successor row.
3. Verify the as-of results against expectations derived from the raw rows, handle listings with no price in that book (LEFT JOIN → NULL), and count active offers per variant.

**Follow-ups:** why the price book is grain `variant × seller × book × interval` (not per listing), type-2 price history for audit, and the sparse-attribute problem (EAV vs JSON column vs wide table — picked per query patterns).

In [0]:
# Solution Q12 — catalog grain chain + effective-dated as-of price join (verified vs raw rows)
from collections import defaultdict
from pyspark.sql import functions as F

listings = [("l1", "p1-v1", "sellerA", "ACTIVE"), ("l2", "p1-v1", "sellerB", "ACTIVE"),
            ("l3", "p1-v2", "sellerB", "ACTIVE"), ("l4", "p2-v1", "sellerC", "PAUSED")]
price_books = [
    ("pb1", "EU_2026", "p1-v1", "sellerA", "EUR", 249.00, "2026-01-01", "2026-06-30"),
    ("pb2", "EU_2026", "p1-v1", "sellerB", "EUR", 239.00, "2026-01-01", "2026-06-30"),
    ("pb3", "EU_2026", "p1-v1", "sellerA", "EUR", 259.00, "2026-07-01", None),        # successor row
    ("pb4", "EU_2026", "p2-v1", "sellerC", "EUR",  89.00, "2026-01-01", None),
    ("pb5", "US_2026", "p1-v1", "sellerA", "USD", 279.00, "2026-01-01", None),
]
l_df = spark.createDataFrame(listings, ["listing_id", "variant_id", "seller_id", "status"])
pb_df = spark.createDataFrame(price_books, ["entry_id", "book", "variant_id", "seller_id",
                                            "currency", "price", "effective_from", "effective_to"])
pb_df = pb_df.withColumn("from", F.to_date("effective_from")).withColumn("to", F.to_date("effective_to"))

# as-of join on 2026-08-01, EU book: pb1 EXPIRED for sellerA -> successor pb3 (259.00) applies
qdate = F.lit("2026-08-01").cast("date")
cond = ((l_df["variant_id"] == pb_df["variant_id"]) & (l_df["seller_id"] == pb_df["seller_id"])
        & (pb_df["book"] == "EU_2026") & (pb_df["from"] <= qdate)
        & ((pb_df["to"].isNull()) | (pb_df["to"] >= qdate)))
asof = (l_df.join(pb_df, cond, "left")
        .select(l_df["listing_id"], l_df["variant_id"], l_df["seller_id"],
                pb_df["price"].alias("price"), pb_df["to"].alias("price_until")))
asof_rows = {r["listing_id"]: (r["price"], r["price_until"]) for r in asof.collect()}

exp_price = {}
for eid, book, var, sel, cur, pr, f_, t_ in price_books:
    if book == "EU_2026" and f_ <= "2026-08-01" and (t_ is None or t_ >= "2026-08-01"):
        exp_price[(var, sel)] = pr
for lid, var, sel, st in listings:
    expect = exp_price.get((var, sel))
    got = asof_rows[lid][0]
    assert (got is None and expect is None) or abs(float(got) - expect) < 1e-9, (lid, got, expect)
assert float(asof_rows["l1"][0]) == 259.0        # expired 249.00 row correctly skipped
assert asof_rows["l3"][0] is None                 # p1-v2/sellerB has no EU_2026 row at all
assert float(asof_rows["l4"][0]) == 89.0          # PAUSED listing still priced (status is offer-level)

offers = {r["variant_id"]: r["sellers"] for r in
          (l_df.where("status = 'ACTIVE'").groupBy("variant_id")
           .agg(F.countDistinct("seller_id").alias("sellers")).collect())}
exp_offers = defaultdict(set)
for _, v, s, st in listings:
    if st == "ACTIVE":
        exp_offers[v].add(s)
assert offers == {v: len(exp_offers[v]) for v in exp_offers}, offers
assert set(offers) == {"p1-v1", "p1-v2"}          # p2-v1 ABSENT: only PAUSED listings (0 actives)
print(f"Q12 OK — as-of prices verified vs raw rows; offers/variant={dict(offers)} (absent key == 0 actives)")

# TALK TRACK
# 1) Ambiguity trap hit live: join on identically-named columns from both sides -> AMBIGUOUS_REFERENCE;
#    qualify every column ref in the join condition and select.
# 2) groupBy semantics: a variant with NO active listings does NOT appear as 0 — it is ABSENT.
#    Dashboards that assume missing groups are zero undercount actives; coalesce explicitly.
# 3) Effective-dating pattern: interval-containment join (from <= as_of AND (to IS NULL OR
#    to >= as_of)); open-ended rows use NULL 'to'. Gap/overlap audits: self-join the book on
#    overlapping intervals per (variant, seller, book) — a GAP silently nulls the price,
#    an OVERLAP silently doubles rows.
# 4) Sparse attributes (p2-v1 missing voltage): wide table is fine at 2 attributes; at 200+,
#    EAV explodes row counts and JSON columns kill pruning — model to the query patterns.

## Q13 — Data Modeling: Experimentation Platform Data Model — Assignment, Exposure, SRM, and Point-in-Time Attribution [HARD]

**Interviewer (LinkedIn, experimentation platform):** Design the core tables for an A/B platform and defend the two classic bugs: **SRM** (sample-ratio mismatch — assigned users who never got exposed) and **point-in-time correctness** (counting a user's revenue from BEFORE they entered the experiment). Build the model, plant both bugs, and detect them with queries.

**Requirements**

1. Tables at the right grains: `assignment` (user × experiment × variant, one row per user), `exposure` (user × experiment × exposed_at), `metric_events` (user × date × revenue).
2. SRM check: count(assigned) vs count(exposed) per experiment — flag leakage (assigned-but-never-exposed).
3. Exposure-gated rollup: revenue counts only on/after the exposure date, per variant. Compare against the NAIVE rollup to quantify the trap.

**Follow-ups:** why assignment and exposure are separate tables at all (assignment is intent, exposure is truth — logging bugs only hit one), hash-based bucketing determinism, and re-randomization/multiple experiments (overlap policies, mutually exclusive tiers).

In [0]:
# Solution Q13 — experimentation model: SRM leakage + exposure-gated attribution (bugs planted & caught)
from collections import defaultdict
from pyspark.sql import functions as F

assignments = [("u1", "exp101", "control",   "2026-11-02 09:00"),
               ("u2", "exp101", "treatment", "2026-11-02 10:00"),
               ("u3", "exp101", "control",   "2026-11-03 09:00"),
               ("u4", "exp101", "treatment", "2026-11-03 11:00"),
               ("u5", "exp101", "control",   "2026-11-04 09:00")]   # BUG 1: never exposed
exposures = [("u1", "exp101", "2026-11-02 09:05"),
             ("u2", "exp101", "2026-11-02 10:02"),
             ("u3", "exp101", "2026-11-03 09:03"),
             ("u4", "exp101", "2026-11-03 11:01")]
metric_events = [("u1", "2026-11-01", 0.0),    # BUG 2: PRE-exposure revenue (would wrongly count)
                 ("u1", "2026-11-03", 12.0),
                 ("u2", "2026-11-03", 20.0),
                 ("u3", "2026-11-04", 8.0),
                 ("u4", "2026-11-04", 15.0),
                 ("u4", "2026-11-10", 3.0),
                 ("u5", "2026-11-05", 50.0)]    # unexposed user's revenue (naive rollup counts it)

asg = spark.createDataFrame(assignments, ["user_id", "exp_id", "variant", "assigned_at"])
exp_ev = spark.createDataFrame(exposures, ["user_id", "exp_id", "exposed_at"])
mev = spark.createDataFrame(metric_events, ["user_id", "metric_date", "revenue"])

# SRM check: assigned vs exposed
leak = (asg.groupBy("exp_id").agg(F.count("*").alias("assigned"))
        .join(exp_ev.groupBy("exp_id").agg(F.count("*").alias("exposed")), "exp_id")
        .withColumn("leakage", F.col("assigned") - F.col("exposed"))).first()
assert leak["assigned"] == 5 and leak["exposed"] == 4 and leak["leakage"] == 1
print(f"SRM check: assigned={leak['assigned']} exposed={leak['exposed']} -> LEAKAGE {leak['leakage']}")

# exposure-gated attribution: metrics only on/after the exposure date
j = (asg.join(exp_ev, ["user_id", "exp_id"])                    # INNER join drops unexposed u5
     .select("user_id", "variant", F.to_date("exposed_at").alias("exp_date")))
gated = (j.join(mev, "user_id")
         .where(F.to_date("metric_date") >= F.col("exp_date"))   # PIT gate
         .groupBy("variant").agg(F.countDistinct("user_id").alias("users"),
                                 F.sum("revenue").alias("revenue")))
res = {r["variant"]: (r["users"], float(r["revenue"])) for r in gated.collect()}

# independent expectations derived straight from the raw rows
gated_exp = defaultdict(float); users_by_var = defaultdict(set)
exp_date = {u: e[:10] for u, _, e in exposures}
variant_of = {u: v for u, _, v, _ in assignments}
for u, d, rev in metric_events:
    if u in exp_date and d >= exp_date[u]:
        gated_exp[variant_of[u]] += rev
for u in exp_date:
    users_by_var[variant_of[u]].add(u)
assert res == {v: (len(users_by_var[v]), gated_exp[v]) for v in gated_exp}, res

naive_d = {r["variant"]: float(r["rev"]) for r in
           (asg.join(mev, "user_id").groupBy("variant")
            .agg(F.sum("revenue").alias("rev")).collect())}
print(f"Q13 OK — gated={res}  naive_trap={naive_d}")
print("  trap quantified: naive control = 70.0 (u1's PRE-exposure 0.0 + u5's unexposed 50.0 wrongly in)")

# TALK TRACK
# 1) Why two tables: assignment = intent (randomizer output), exposure = truth (what shipped).
#    A client logging bug hits exposure only — the SRM query above catches it as leakage. In
#    production the check is chi-square on expected vs observed split (day 26's Wilson/Benford
#    toolkit generalizes: expected 50/50, observed 5/4 -> p-value gate).
# 2) PIT gate: metric_date >= exposure date. u1's 2026-11-01 revenue (0.0) predates exposure —
#    naive join counts it; gated join drops it. u5 disappears via the INNER join to exposure.
# 3) Grains: assignment is one row per user per experiment (re-randomization is a NEW
#    experiment, never an update — keeps replay semantics); metric_events are user-day facts.
# 4) Scale-up: exposure-gated attribution is a range join; at billions of events, bucket
#    users by exposure date (day-partitioned) and pre-filter events to the window.

## Q14 — Snowflake: External Network Access — Network Rules, API Integrations, Secrets, and Governed UDFs [HARD]

**Interviewer (Netflix, platform security):** A fraud-scoring UDF must call a third-party API, but analysts must not be able to exfiltrate data or read credentials. Walk the full Snowflake **external access** governance stack and prove the enforcement semantics.

**Requirements**

1. State the four-layer stack and their order: **network rule** (egress allowlist host:port) → **API integration** (allowed URL prefixes + auth) → **external access integration** (binds network rules + secrets) → **UDF declaration** referencing the integration and secrets.
2. Prove enforcement with a mock: an allowlisted call enriches; a non-allowlisted domain is BLOCKED; the secret never appears in shipped code or logs.
3. Reference DDL for each object (runnable in Snowflake, not here).

**Follow-ups:** why the token lives in a Snowflake SECRET and not in the function body (SCM scanning, log leakage, rotation), external access vs external functions (AWS Lambda path, deprecated for new work), and how this compares to Databricks' secrets + UC connections pattern.

In [0]:
# Solution Q14 — governed enrichment: network rule -> API integration -> secret -> UDF (mocked, enforced)
SNOWFLAKE_DDL = """
-- Layer 1: NETWORK RULE — the ONLY hosts a UDF may reach
CREATE OR REPLACE NETWORK RULE corp.outbound_fraud_api_rule
  MODE = EGRESS  TYPE = HOST_PORT
  VALUE_LIST = ('api.fraudcheck.example:443');

-- Layer 2: API INTEGRATION — allowed URL prefixes + auth model
CREATE OR REPLACE API INTEGRATION corp.fraud_api_integration
  API_PROVIDER = 'custom_api_client'
  API_ALLOWED_PREFIXES = ('https://api.fraudcheck.example/')
  ALLOWED_AUTHENTICATION_SECRETS = (SECRET);

-- Layer 3: EXTERNAL ACCESS INTEGRATION — binds rules + secrets; the UDF's permission
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION corp.fraud_ext_integration
  ALLOWED_NETWORK_RULES = (corp.outbound_fraud_api_rule)
  ALLOWED_AUTHENTICATION_SECRETS = (corp.fraud_api_token);

-- Layer 4: the SECRET — token lives server-side, referenced BY NAME in the UDF
CREATE OR REPLACE SECRET corp.fraud_api_token
  TYPE = GENERIC_STRING;

-- The UDF: references the integration + secret; code below can only call allowlisted hosts
CREATE OR REPLACE FUNCTION corp.fraud_score(card_id STRING)
  RETURNS FLOAT  LANGUAGE PYTHON  RUNTIME_VERSION = '3.10'  HANDLER = 'score'
  EXTERNAL_ACCESS_INTEGRATIONS = (corp.fraud_ext_integration)
  SECRETS = ('token' = corp.fraud_api_token)
  AS $$ ... requests.get('https://api.fraudcheck.example/v1/score',
                         headers={'Authorization': f'Bearer {_secrets['token']}}) ... $$;
"""

class MockSecrets:
    """Stands in for Snowflake secrets: the value NEVER appears in UDF code or output."""
    _store = {"corp.fraud_api_token": "tk_9f3e-secret-value"}
    def get(self, name):
        return self._store[name]

ALLOWED_PREFIXES = ("https://api.fraudcheck.example/",)

def snowflake_udf_enrich(card_id, target_url, secrets=MockSecrets()):
    # gate 1: NETWORK RULE — only allowlisted egress prefixes pass (enforced by the platform)
    if not any(target_url.startswith(p) for p in ALLOWED_PREFIXES):
        return {"card_id": card_id, "status": "BLOCKED_BY_NETWORK_RULE", "score": None}
    token = secrets.get("corp.fraud_api_token")   # gate 2: secret via integration, never hardcoded
    score = (hash((card_id, "salt")) % 1000) / 10.0   # deterministic stand-in for the API call
    return {"card_id": card_id, "status": "OK", "score": score}

ok = snowflake_udf_enrich("card_1", "https://api.fraudcheck.example/v1/score?card=card_1")
blocked = snowflake_udf_enrich("card_2", "https://evil.example.exfiltrate?t=card_2")
assert ok["status"] == "OK" and ok["score"] is not None
assert blocked["status"] == "BLOCKED_BY_NETWORK_RULE" and blocked["score"] is None
assert "tk_" not in SNOWFLAKE_DDL            # no literal token anywhere in shipped code
assert "tk_" not in repr(ok) and "tk_" not in repr(blocked)
print("Q14 OK — allowlisted call enriched; exfil domain blocked; secret never in code or results")

# TALK TRACK
# 1) Order of the stack matters: the UDF references the EXTERNAL ACCESS INTEGRATION; that
#    integration is itself bounded by the NETWORK RULE (where) and the SECRETS (credentials).
#    An analyst with USAGE on the function but no ownership of the integration can neither
#    broaden the allowlist nor read the token.
# 2) Why secrets beat embedding: SCM history, log scrubbing, rotation without code changes,
#    and blast radius (leaked function source leaks nothing).
# 3) Compare/contrast (say this in a Databricks interview): same goals via Databricks secrets
#    scopes / UC credential vending + network policies on serverless; Snowflake's integration
#    objects make the allowlist a first-class governance artifact.
# 4) Legacy path: EXTERNAL FUNCTIONS over AWS Lambda/Azure functions — now superseded by
#    external access for new work; API integrations remain for OAuth-based provider-specific
#    flows (e.g., GitHub, PagerDuty integrations).

## Q15 — Snowflake: Warehouse Sizing Under Load — Queueing Theory, the Hockey Stick, and Scale-Up vs Scale-Out [FAANG]

**Interviewer (Apple, BI platform):** Queries arrive randomly; a warehouse has finite concurrency slots. Size too small and waits explode; size too big and you burn credits. Simulate M/M/1 and M/M/c queues and answer: what does the wait-vs-utilization curve look like, when do extra warehouses beat one bigger one — and when does the bigger one win anyway?

**Requirements**

1. Simulate M/M/1 (Poisson arrivals, exponential service) at ρ = 0.5, 0.8, 0.95; show mean queue wait and assert the hockey stick (0.95 wait > 5× the 0.5 wait) plus agreement with theory `E[Wq] = ρ/(μ−λ)`.
2. Verify **Little's law** `L = λW` on a run.
3. At the same total capacity (λ=3.2, capacity 4.0): compare ONE fast server (μ=4) vs FOUR slow servers (μ=1 each). Assert BOTH directions: 4 warehouses give lower QUEUE wait; 1 big warehouse gives lower END-TO-END time. Explain the Snowflake sizing lesson.

**Follow-ups:** why multi-cluster warehouses exist (queue wait SLAs, not throughput), warehouse auto-suspend interaction with arrival bursts (cold-start minutes), and where this model breaks (non-exponential, heterogeneous query sizes — tail of the service distribution dominates).

In [0]:
# Solution Q15 — M/M/1 and M/M/c queueing: hockey stick, Little's law, scale-up vs scale-out
import numpy as np
import heapq

def simulate_mm1(lam, mu, n=60_000, seed=42, warmup=5_000):
    rng = np.random.default_rng(seed)
    waits, at, service_start = [], 0.0, 0.0
    for i in range(n):
        at += rng.exponential(1 / lam)
        svc = rng.exponential(1 / mu)
        start = max(at, service_start)
        service_start = start + svc
        if i >= warmup:
            waits.append(start - at)
    return np.array(waits)

# (1) hockey stick
results = {rho: simulate_mm1(rho, 1.0).mean() for rho in (0.5, 0.8, 0.95)}
print("mean queue wait by utilization:", {k: round(v, 3) for k, v in results.items()})
assert results[0.95] > 5 * results[0.5]                       # explosion past 90%
assert abs(results[0.95] - (1 / 0.05 - 1)) / (1 / 0.05) < 0.15   # theory E[W]+E[S]=1/(mu-lam)

# (2) Little's law: L = lambda * W
w = simulate_mm1(0.8, 1.0, seed=7)
W = w.mean() + 1.0                      # time in system = wait + service
L = 0.8 * W
assert 0.7 < L < 5.0 and abs(L - 0.8 / 0.2) / 4.0 < 0.25     # theory L = rho/(1-rho) = 4
print(f"Q15a OK — hockey stick verified; Little's law L={L:.2f} (theory 4.0)")

def simulate_mmc(lam, mu_per_server, c, n=80_000, seed=3, warmup=8_000):
    rng = np.random.default_rng(seed)
    servers = [0.0] * c
    heapq.heapify(servers)                 # earliest-free server first (FCFS)
    waits, at = [], 0.0
    for i in range(n):
        at += rng.exponential(1 / lam)
        start = max(heapq.heappop(servers), at)
        heapq.heappush(servers, start + rng.exponential(1 / mu_per_server))
        if i >= warmup:
            waits.append(start - at)
    return np.array(waits)

# (3) scale-up vs scale-out at equal TOTAL capacity (lam=3.2 of capacity 4.0 -> rho=0.8)
lam = 3.2
big_w   = simulate_mmc(lam, mu_per_server=4.0, c=1, seed=11)   # one big warehouse
small_w = simulate_mmc(lam, mu_per_server=1.0, c=4, seed=13)   # four small warehouses
big_resp   = big_w.mean() + 1 / 4.0     # + service on the fast server
small_resp = small_w.mean() + 1 / 1.0   # + service on a small server
print(f"queue wait:  1-big {big_w.mean():.3f} vs 4-small {small_w.mean():.3f}")
print(f"end-to-end:  1-big {big_resp:.3f} vs 4-small {small_resp:.3f}")
assert small_w.mean() < big_w.mean()    # more warehouses -> less QUEUEING
assert big_resp < small_resp            # bigger warehouse -> faster COMPLETION
assert abs(big_w.mean() - 1.0) / 1.0 < 0.15   # M/M/1(mu=4) theory E[Wq] = lam/(mu(mu-lam)) = 1.0
print("Q15b OK — multi-cluster cuts queue wait; single big warehouse cuts end-to-end latency")

# TALK TRACK
# 1) The hockey stick is THE sizing argument: 0.5 -> 0.95 utilization moved mean wait by >5x,
#    and it diverges as rho -> 1 (E[Wq] = rho/(mu-lam)). Snowflake sizing is queue math: target
#    utilization, not headroom-free averages. Practical: keep steady-state under ~70-80%.
# 2) The counterintuitive pair we PROVED (theory said so, sim agreed):
#    - 4 small warehouses BEAT 1 big one on QUEUE WAIT (parallelism absorbs bursts)
#      -> when analysts complain about queries WAITING, add warehouses (multi-cluster),
#         don't just upsize.
#    - 1 big warehouse BEATS 4 small on END-TO-END time (each query runs 4x faster)
#      -> when single long queries are slow, UPSIZE.
#    Scale-out serves concurrency SLAs; scale-up serves latency. Both at equal credits.
# 3) Model limits to volunteer: real warehouses have heterogeneous query sizes (heavy-tailed
#    service — waits are WORSE than exponential predicts), auto-suspend adds cold-start when
#    bursts arrive at a suspended warehouse, and credit cost is not linear in size (halving
#    wall time costs more than 2x credits sometimes — verify against the size-vs-credit table).

## Q16 — Monitoring & Orchestration: DAG Critical Path, Slack, and Backfill Math [FAANG]

**Interviewer (Amazon, supply-chain pipelines):** The nightly gold layer has a 90-minute SLA. Given the task DAG with durations, tell me: the critical path, how much slack each task has before it threatens the SLA, whether 2 parallel job slots meet it when 1 does not, and what the makespan floor is no matter how many slots you add.

**Requirements**

1. Compute per-task earliest finish (topological DP), the **critical path**, and **total float** (latest start − earliest start) via a reverse pass; verify float with a property test — delaying a task BY its float keeps the makespan, by float+1 breaks it.
2. Show slot math: 1 slot (fully serial), 2 slots, 4+ slots — and state the floor (the critical path) that no parallelism beats.

**Follow-ups:** how this drives backfill windows (a task with 30 min float can absorb a 30-min upstream delay), why `Run If`/dependency design should mirror the DATA DAG not the org chart, and critical-path monitoring (alert on the path tasks, not everything).

In [0]:
# Solution Q16 — critical path, total float (property-verified), and slot math
tasks = {  # task: (duration_min, dependencies)
    "ingest_orders":   (20, []),
    "ingest_users":    (15, []),
    "validate":        (10, ["ingest_orders", "ingest_users"]),
    "dim_users":       (25, ["ingest_users"]),
    "fact_orders":     (40, ["validate"]),
    "gold_dashboard":  (15, ["fact_orders", "dim_users"]),
    "gold_exports":    (10, ["fact_orders"]),
}
order = ["ingest_orders", "ingest_users", "validate", "dim_users",
         "fact_orders", "gold_dashboard", "gold_exports"]
pos = {t: i for i, t in enumerate(order)}
assert all(pos[d] < pos[t] for t, (dur, deps) in tasks.items() for d in deps)  # valid topo order

# earliest finish (forward DP) -> critical path
finish = {}
for t in order:
    dur, deps = tasks[t]
    finish[t] = dur + max((finish[d] for d in deps), default=0)
critical_len = max(finish.values())
critical_path = []
cur = max(finish, key=finish.get)
while True:
    critical_path.append(cur)
    dur, deps = tasks[cur]
    if not deps:
        break
    cur = max(deps, key=lambda d: finish[d])       # walk the binding predecessor
critical_path.reverse()                               # present root -> sink
# verify the path against ALL root-to-sink paths
all_paths = []
def walk(t, path=()):
    dur, deps = tasks[t]
    if not deps:
        all_paths.append(path + (t,))
    for d in deps:
        walk(d, path + (t,))
walk("gold_dashboard")
best = max((sum(tasks[t][0] for t in p), p) for p in all_paths)
assert critical_len == best[0] == 85
assert tuple(critical_path) == best[1] or tuple(critical_path) == tuple(reversed(best[1]))

# total float via reverse pass (latest start - earliest start)
def compute_slack(tasks, order, deadline):
    latest = {}
    for t in reversed(order):
        dur, _ = tasks[t]
        succ_starts = [latest[s] for s in tasks if t in tasks[s][1]]
        latest_finish = min(succ_starts) if succ_starts else deadline
        latest[t] = latest_finish - dur
    return {t: latest[t] - (finish[t] - tasks[t][0]) for t in tasks}
slack85 = compute_slack(tasks, order, critical_len)   # vs project-completion deadline
assert sorted(t for t in slack85 if slack85[t] == 0) == ["fact_orders", "gold_dashboard",
                                                          "ingest_orders", "validate"]

# makespan with N parallel slots (list scheduling, ready-ordered)
def makespan(task_map, slots, order):
    free = [0.0] * slots
    done_at = {}
    for t in order:
        dur, deps = task_map[t]
        ready = max((done_at[d] for d in deps), default=0)
        i = min(range(slots), key=lambda k: max(free[k], ready))
        start = max(free[i], ready)
        done_at[t] = start + dur
        free[i] = done_at[t]
    return max(done_at.values())

mk_inf, mk1, mk2 = makespan(tasks, 8, order), makespan(tasks, 1, order), makespan(tasks, 2, order)
assert mk_inf == 85                     # enough slots: the critical path IS the floor
assert mk1 == sum(d for d, _ in tasks.values()) == 135   # 1 slot = fully serial
assert mk2 == 85                        # 2 slots already meet the 90-min SLA

# PROPERTY check of float: +float keeps makespan, +float+1 breaks it
for t, float_ in slack85.items():
    up = {k: ((v[0] + float_, v[1]) if k == t else v) for k, v in tasks.items()}
    up1 = {k: ((v[0] + float_ + 1, v[1]) if k == t else v) for k, v in tasks.items()}
    assert makespan(up, 8, order) == 85 and makespan(up1, 8, order) > 85, (t, float_)

print(f"Q16 OK — critical path {critical_path} = {critical_len} min")
print(f"  makespan: 1 slot={mk1}  2 slots={mk2}  4+ slots={mk_inf} (floor = critical path)")
print(f"  float: {slack85}  (property-verified: +float holds, +float+1 breaks)")

# TALK TRACK
# 1) Reverse pass TRAP hit live twice: the latest-allowed FINISH for t is the successor's
#    latest START (no extra subtraction) — subtracting the successor's duration silently
#    corrupts every float. Hand-derived floats were wrong twice; the property test caught it.
#    Property-based verification > eyeballing arithmetic.
# 2) Float is your backfill currency: dim_users has 30 min of float, so a 25-min upstream
#    delay is survivable; zero-float tasks are where pager alerts belong (critical-path
#    monitoring, not blanket monitoring).
# 3) Slot math: 1 slot = serial sum (135); the 90-min SLA needs exactly 2 slots here — 85 min
#    make vs 90 SLA is only 5 min of headroom, i.e. ONE late ingest eats it. State the floor:
#    more slots never beats the critical path; only shortening path tasks (or removing deps)
#    moves the floor.
# 4) Databricks mapping: slots = concurrent task slots in a job; 'Run If' + dependencies
#    should mirror the data lineage DAG — org-chart-shaped orchestration wastes float.

In [0]:
    if not deps:
        break
    cur = max(deps, key=lambda d: finish[d])       # walk the binding predecessor
critical_path.reverse()                               # present root -> sink

## Q17 — Monitoring: Tail-Latency SLOs for Data Products — Percentiles, Tiering, and Regression Detection [FAANG]

**Interviewer (Uber, serving infra):** Two data products share a serving platform: a tier-1 search API (p99 ≤ 300 ms SLA) and tier-2 batch reports (p99 ≤ 1500 ms). Build the SLO report: percentiles, tier gates, and a day-over-day regression rule — then prove it flags the product that regressed while letting the healthy one pass.

**Requirements**

1. From synthetic serving logs per product (body + tail distributions), compute p50/p95/p99.
2. Tier gates: p99 vs tier SLO → PASS/BREACH. Regression: p99 > 1.5× yesterday's p99 → REGRESSION.
3. Assert: tier1 (planted bad tail: 20 of 500 requests at 400–600 ms) → BREACH + REGRESSION; tier2 → PASS + stable; percentiles strictly ordered.

**Follow-ups:** why p99 and not max (single worst request), SLO burn-rate alerting (fast-burn vs slow-burn windows), and multi-window alerting to catch both sudden and slow degradations.

In [0]:
# Solution Q17 — tiered SLO report: percentiles + tier gate + day-over-day regression
import numpy as np

def serving_day(seed, n_body, body_lo, body_hi, n_tail, tail_lo, tail_hi):
    r = np.random.default_rng(seed)
    body = r.uniform(body_lo, body_hi, n_body)
    tail = r.uniform(tail_lo, tail_hi, n_tail)
    return np.sort(np.concatenate([body, tail]))

today = {
    "tier1_search": serving_day(1, 480, 20, 80, 20, 400, 600),       # planted bad tail (4% >= 400ms)
    "tier2_reports": serving_day(2, 480, 300, 900, 20, 1200, 1400), # normal day
}
yesterday = {
    "tier1_search": serving_day(3, 480, 20, 80, 20, 250, 320),       # normal day
    "tier2_reports": serving_day(4, 480, 300, 900, 20, 1200, 1400),
}
SLO = {"tier1_search": {"p99": 300}, "tier2_reports": {"p99": 1500}}

def pct(a, q):
    return float(np.percentile(a, q))

report = []
for prod in today:
    lats = today[prod]
    p50, p95, p99 = pct(lats, 50), pct(lats, 95), pct(lats, 99)
    y99 = pct(yesterday[prod], 99)
    report.append((prod, round(p50, 1), round(p95, 1), round(p99, 1), round(y99, 1),
                   "PASS" if p99 <= SLO[prod]["p99"] else "BREACH",
                   "REGRESSION" if p99 > 1.5 * y99 else "stable"))

for row in report:
    print(f"{row[0]:14} p50={row[1]:7} p95={row[2]:7} p99={row[3]:7} "
          f"y-p99={row[4]:7} -> {row[5]:7} {row[6]}")

r1 = next(r for r in report if r[0] == "tier1_search")
r2 = next(r for r in report if r[0] == "tier2_reports")
assert r1[5] == "BREACH" and r1[6] == "REGRESSION", r1
assert r2[5] == "PASS" and r2[6] == "stable", r2
assert r1[3] > r1[2] > r1[1] and r2[3] > r2[2] > r2[1]
print("Q17 OK — tier1 tail caught (BREACH + REGRESSION); tier2 PASS + stable; percentiles ordered")

# TALK TRACK
# 1) Tail design matters: 3 outliers in 500 moved p99 NOT AT ALL (0.6% < the 1% the 99th
#    percentile even sees). We needed 20/500 (4%) planted at 400-600ms to breach 300. When you
#    demo an SLO gate, engineer the tail to actually cross the threshold — and when you
#    debug one, remember p99 is INVISIBLE to anything affecting < 1% of traffic.
# 2) p99 over max: one pathological request (GC pause, retry storm) would page you forever;
#    p99 tolerates 1% badness by construction. Burn-rate alerting generalizes: alert when
#    error-budget CONSUMPTION rate exceeds plan over 1h (fast burn, page) vs 6h (slow burn,
#    ticket).
# 3) Regression rule (p99 > 1.5x yesterday) catches slow drift the absolute SLO misses — a
#    product can PASS its SLO while degrading week over week until it suddenly doesn't.
# 4) Databricks mapping: this report is a metrics-table + SQL alert (system tables / Lakehouse
#    Monitoring store the latency; an alert evaluates the query on a schedule).

## Q18 — Monitoring: Auto-Triage Bot — Freshness Alert → Lineage Walk → Root Cause [HARD]

**Interviewer (Meta, data platform on-call tooling):** The dashboard-mart table is stale and a pager fired. On-call currently walks lineage by hand across 7 tables and 6 jobs. Build the triage algorithm the bot runs: from the stale table, walk the upstream lineage, separate stale from fresh ancestors, and finger the failed job at the root — with a deterministic, explainable report.

**Requirements**

1. Inputs: `lineage` (upstream → downstream edges), `freshness` (minutes since last success per table), `job_runs` (writer job → status/age per table). Scenario: `job_silver_events` FAILED 240 min ago; everything downstream is stale-by-cascade; `job_silver_users` succeeded later so `silver_users` looks STALE by age but its writer is healthy.
2. Algorithm: ancestor walk from the alerting table; stale set = freshness > 60 min; root cause = the FAILED writer of the MOST-UPSTREAM stale table (max age among failed writers).
3. Assert the bot names exactly `job_silver_events` on `silver_events`, and report the walked ancestor set and stale set.

**Follow-ups:** why “most-upstream failed writer” beats “most-recent failure” (downstream SKIPPED jobs are victims, not causes), the explainability requirement (the bot must print the WHY chain), and how this wires into Databricks (system tables lineage + job run history + a Genie/agent action).

In [0]:
# Solution Q18 — auto-triage: ancestor walk + freshness classification + root-cause job
from collections import defaultdict

lineage = {  # upstream -> [downstreams]
    "bronze_events": ["silver_events"],
    "bronze_users":   ["silver_users", "silver_events"],
    "silver_events":  ["gold_activity"],
    "silver_users":   ["gold_activity", "gold_profiles"],
    "gold_activity":  ["mart_dashboard"],
    "gold_profiles":  ["mart_dashboard"],
}
freshness = {  # minutes since last SUCCESSFUL refresh
    "bronze_events": 15, "bronze_users": 12, "silver_events": 240, "silver_users": 200,
    "gold_activity": 235, "gold_profiles": 195, "mart_dashboard": 230,
}
job_runs = {  # (writer job, table it writes): (status, minutes_ago)
    ("job_ingest_events", "bronze_events"): ("SUCCEEDED", 15),
    ("job_ingest_users", "bronze_users"):   ("SUCCEEDED", 12),
    ("job_silver_events", "silver_events"): ("FAILED", 240),      # the ROOT CAUSE
    ("job_silver_users", "silver_users"):   ("SUCCEEDED", 45),    # ran later; stale-by-age is a TRAP
    ("job_gold", "gold_activity"):           ("SKIPPED", 235),      # victim, not cause
    ("job_gold", "gold_profiles"):           ("SKIPPED", 235),      # victim, not cause
}
SLA_MIN = 60

def triage(stale_table):
    # invert lineage: for each table, its parents
    parents = defaultdict(list)
    for u, downs in lineage.items():
        for d in downs:
            parents[d].append(u)
    # ancestor walk from the alerting table
    ancestors, stack = {stale_table}, [stale_table]
    while stack:
        t = stack.pop()
        for p in parents.get(t, []):
            if p not in ancestors:
                ancestors.add(p)
                stack.append(p)
    stale_anc = [t for t in ancestors if freshness[t] > SLA_MIN]
    # root cause: FAILED writer of the MOST-UPSTREAM stale table (largest minutes_ago)
    candidates = [(job, tbl, ago) for (job, tbl), (status, ago) in job_runs.items()
                  if status == "FAILED" and tbl in stale_anc]
    if not candidates:
        return ancestors, stale_anc, None
    job, tbl, ago = min(candidates, key=lambda c: -c[2])
    return ancestors, stale_anc, (job, tbl)

anc, stale_anc, cause = triage("mart_dashboard")
assert cause == ("job_silver_events", "silver_events"), cause
assert anc == {"mart_dashboard", "gold_activity", "gold_profiles", "silver_events",
               "silver_users", "bronze_events", "bronze_users"}, anc

print("AUTO-TRIAGE REPORT — alert: mart_dashboard stale")
print(f"  lineage walked: {len(anc)} ancestors")
print(f"  stale (> {SLA_MIN} min): {sorted(stale_anc)}")
print(f"  ROOT CAUSE: {cause[0]} writing {cause[1]} — FAILED, blocking {len(stale_anc) - 1} downstream tables")
print("  why not job_silver_users? silver_users IS stale (200 min) but its writer SUCCEEDED 45 min")
print("  ago — stale-by-age, not failed. Why not job_gold? SKIPPED = victim of the upstream failure.")
print("Q18 OK — root cause = job_silver_events on silver_events")

# TALK TRACK
# 1) Victim-vs-cause discipline: SKIPPED jobs and stale-but-successfully-written tables are
#    CASCADE damage. Ranking by 'most-upstream FAILED writer' (max age among failures) picks
#    the blocking node; ranking by 'most recent failure' would page job_gold instead.
# 2) Explainability is a requirement, not a nicety: the bot prints the ancestor set, the stale
#    set, and the why-not chain — on-call trusts it only while its reasoning is auditable.
# 3) Databricks wiring: lineage from system tables (system.access.lineage / table lineage in
#    UC), job history from system.lakeflow (job/run tables), freshness from
#    _commit timestamps or Lakehouse Monitoring metrics; the bot becomes a scheduled notebook
#    that the pager webhook triggers, ending in a Slack message with this exact report.
# 4) Extending: multi-root scenarios (two independent failures) -> return ALL failed
#    writers whose tables are not downstream of another failed writer; add a 're-run
#    suggestion' action (repair run from the root cause's job, not the alerting table's).

## Day 27 — Scorecard

| # | Category | Topic | Tag | Verified via |
|---|----------|-------|-----|--------------|
| Q1 | SQL Advanced | Pareto + Gini (Lorenz trapezoid) | [FAANG] | closed-form Gini agreement to 1e-9 |
| Q2 | SQL Advanced | Second-price auction + budget pacing | [FAANG] | all five auction edge cases asserted |
| Q3 | Python | Fenwick tree + inversion count | [FAANG] | brute-force O(n²) + 2000-element stress test |
| Q4 | Python | Rabin-Karp rolling hash | [FAANG] | str.find cross-check + 4000-char scan |
| Q5 | PySpark | Distributed bootstrap CIs | [FAANG] | point-in-CI + delta-method width (±40%) |
| Q6 | PySpark | Distributed PageRank + dangling mass | [FAANG] | pure-Python reference to 1e-4, Σrank=1 |
| Q7 | Spark Deep | Catalyst logical optimizer forensics | [HARD] | plan text + independent row recount |
| Q8 | Spark Deep | Join strategy + exchange selection | [HARD] | three plans, counts invariant across hints |
| Q9 | ETL | INT96/timezones/ANSI parser | [HARD] | offset equality, tz flip, Parquet round-trip |
| Q10 | ETL | Voyager migration + parity gates | [FAANG] | score derivation + 3 gate paths |
| Q11 | ETL | Kappa replay + dual-run | [HARD] | derived v1/v2, determinism, late append |
| Q12 | Data Modeling | Marketplace catalog + price book | [FAANG] | as-of expectations from raw rows |
| Q13 | Data Modeling | Experimentation platform model | [HARD] | SRM leakage=1; gated vs naive rollups |
| Q14 | Snowflake | External network access + secrets | [HARD] | allowlist gate + exfil block + no-token leak |
| Q15 | Snowflake | Warehouse queueing theory | [FAANG] | hockey stick, Little's law, M/M/1 vs M/M/4 |
| Q16 | Monitoring | DAG critical path + float + slots | [FAANG] | all-paths brute force + float property test |
| Q17 | Monitoring | Tail-latency SLO report | [FAANG] | planted tail flagged, healthy tier passes |
| Q18 | Monitoring | Auto-triage root cause | [HARD] | exact root-cause job + full ancestor set |

**Debug lessons banked this session:** mapInPandas takes an ITERATOR of DataFrames (Spark 4) · bootstrap UDF must regenerate base data bit-identically to the driver or the CI describes different data · `numpy.var`'s 2nd positional arg is `axis`, not `ddof` · join-column ambiguity needs qualified refs · groupBy drops zero-groups entirely · 1e-9 convergence tolerances time out Spark Connect loops · hand-derived float arithmetic loses to property-based tests · plan text (isFinalPlan=false) can mislead — recount rows · a stray comma before an assignment is still a syntax error.